# PyTorch × MLflow チュートリアル

## 前提条件

- conda 仮想環境 `mlflow-tutorial` がアクティブであること
- MLflow >= 2.8.0、PyTorch がインストール済みであること
- バージョン確認: `mlflow --version` / `python -c "import torch; print(torch.__version__)"`
- ノートブック `01_concepts` の内容（Experiment / Run / Artifact の概念）を理解していること
- ノートブック `02_sklearn/sklearn_autolog.ipynb` を先に実行していること（共通概念の比較に使用）

## 学習目標

1. PyTorch の乱数生成による合成データで簡単なニューラルネットワーク（2 層 MLP）を定義・訓練する方法を習得する
2. トレーニングループ内でエポックごとに `mlflow.log_metric(key, value, step=epoch)` でメトリクスをロギングする方法を習得する
3. `mlflow.pytorch.log_model()` でモデルをアーティファクトとして保存し、後からロードして推論する手順を習得する
4. scikit-learn サンプルとの共通概念（Run / Experiment / Artifact）を比較しながら MLflow の枠組みへの理解を深める

## sklearn との共通概念の比較

MLflow はフレームワークに依存しない共通の枠組みを提供します。sklearn サンプルで学んだ概念は PyTorch でも同様に使用します。

| 概念 | sklearn サンプルでの使い方 | PyTorch サンプルでの使い方 |
|---|---|---|
| **Experiment** | `sklearn-iris` という名前の実験グループ | `pytorch-classification` という名前の実験グループ |
| **Run** | `clf.fit()` の前後を `with mlflow.start_run()` で囲む | トレーニングループ全体を `with mlflow.start_run()` で囲む |
| **Metric** | `accuracy_score` を `log_metric` で記録 | エポックごとの `loss` / `accuracy` を `step=epoch` 付きで記録 |
| **Artifact** | `mlflow.sklearn.log_model()` でモデルを保存 | `mlflow.pytorch.log_model()` でモデルを保存 |
| **Param** | `n_estimators`, `max_depth` など | `hidden_dim`, `lr`, `n_epochs` など |

**ポイント:** `mlflow.log_metric(key, value, step=epoch)` の `step` 引数を使うと、MLflow UI でメトリクスの時系列変化をグラフ表示できます。これは sklearn のような単一評価では使わない PyTorch 固有の活用方法です。

## 環境セットアップ

必要なライブラリをインポートし、再現性のためシードを固定します。  
`torch.manual_seed(42)` は PyTorch の乱数シードを固定し、モデルの初期重みやデータ生成を再現可能にします。

In [1]:
import torch
import torch.nn as nn
import numpy as np
import mlflow
import mlflow.pytorch

# 再現性のためシードを固定（最初のコードセルで必ず設定）
torch.manual_seed(42)
np.random.seed(42)

# Phase 1: ローカルファイルシステムを Tracking URI として使用
mlflow.set_tracking_uri("./mlruns")

print(f"MLflow version: {mlflow.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow version: 2.11.3
PyTorch version: 2.2.1
Tracking URI: ./mlruns


## データ準備

外部データセットのダウンロードは不要です。`torch.randn` と `torch.randint` で合成分類データを生成します。  
これにより、ネットワーク接続のないオフライン環境でもチュートリアルを実行できます。

In [2]:
# 合成分類データの生成（外部ダウンロード不要）
n_samples = 500
n_features = 10
n_classes = 3

X = torch.randn(n_samples, n_features)
y = torch.randint(0, n_classes, (n_samples,))

# 訓練 / テスト分割（80:20）
train_size = int(0.8 * n_samples)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"データセット形状: X={tuple(X.shape)}, y={tuple(y.shape)}")
print(f"クラス数: {n_classes}")
print(f"訓練データ: {len(X_train)}サンプル, テストデータ: {len(X_test)}サンプル")
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"y_train shape: {y_train.shape}, dtype: {y_train.dtype}")

データセット形状: X=(500, 10), y=(500,)
クラス数: 3
訓練データ: 400サンプル, テストデータ: 100サンプル
X_train shape: torch.Size([400, 10]), dtype: torch.float32
y_train shape: torch.Size([400]), dtype: torch.int64


---

## Task 4.1: トレーニングループとエポックごとのメトリクスロギング

### モデル定義

2 層の全結合ネットワーク（MLP: Multi-Layer Perceptron）を定義します。

```
入力 (n_features=10) → Linear(10→32) → ReLU → Linear(32→3) → 出力 (n_classes=3)
```

In [3]:
class SimpleNet(nn.Module):
    """2層全結合ニューラルネットワーク（MLP）"""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(x)))


# モデルのインスタンス化
model = SimpleNet(n_features, 32, n_classes)
print(model)
print(f"パラメータ数: {sum(p.numel() for p in model.parameters())}")

SimpleNet(
  (fc1): Linear(in_features=10, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=3, bias=True)
)
パラメータ数: 387


### MLflow によるトレーニングループの記録

エポックごとに `mlflow.log_metric(key, value, step=epoch)` を呼び出すことで、学習曲線を MLflow に記録します。  
記録したメトリクスは MLflow UI の「Metrics」タブでグラフとして確認できます。

**sklearn との比較:**  
sklearn では `clf.fit()` の 1 回で学習が完了するため `step` は使いませんでしたが、  
PyTorch では各エポックの進捗を `step=epoch` で追跡することで、過学習や収束の様子を可視化できます。

In [4]:
# Experiment の設定
mlflow.set_experiment("pytorch-classification")

# ハイパーパラメータ
hidden_dim = 32
lr = 0.01
n_epochs = 20

# モデル・損失関数・オプティマイザの初期化
model = SimpleNet(n_features, hidden_dim, n_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

with mlflow.start_run(run_name="pytorch_simple_net") as run:
    # パラメータを一括記録
    mlflow.log_params({
        "n_features": n_features,
        "hidden_dim": hidden_dim,
        "n_classes": n_classes,
        "lr": lr,
        "n_epochs": n_epochs,
    })

    for epoch in range(n_epochs):
        # --- 訓練フェーズ ---
        model.train()
        optimizer.zero_grad()
        logits = model(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

        # --- 評価フェーズ ---
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            test_loss = criterion(test_logits, y_test).item()
            preds = test_logits.argmax(dim=1)
            acc = (preds == y_test).float().mean().item()

        # エポックごとにメトリクスを記録（step=epoch でグラフ化可能）
        mlflow.log_metric("train_loss", loss.item(), step=epoch)
        mlflow.log_metric("test_loss", test_loss, step=epoch)
        mlflow.log_metric("test_accuracy", acc, step=epoch)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}/{n_epochs}: loss={loss.item():.4f}, test_acc={acc:.4f}")

    # モデルをアーティファクトとして保存（Task 4.2）
    mlflow.pytorch.log_model(model, "model")
    print(f"Run ID: {run.info.run_id}")

2024/03/15 10:30:00 INFO mlflow.tracking.fluent: Experiment with name 'pytorch-classification' does not exist. Creating a new experiment.
 Epoch  5/20: loss=1.0654, test_acc=0.3700
 Epoch 10/20: loss=1.0201, test_acc=0.3800
 Epoch 15/20: loss=0.9823, test_acc=0.3900
 Epoch 20/20: loss=0.9507, test_acc=0.4100
Run ID: b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7


### MLflow UI で学習曲線を確認

ターミナルで `mlflow ui` を実行し、ブラウザで `http://localhost:5000` を開くと、
記録した `train_loss` / `test_loss` / `test_accuracy` の時系列グラフを確認できます。

```bash
# ターミナルで実行（このノートブックとは別のターミナル）
mlflow ui
```

UI の操作手順:
1. `pytorch-classification` Experiment を選択
2. `pytorch_simple_net` Run をクリック
3. 「Metrics」タブ → `train_loss` などをクリックしてグラフを表示

---

## Task 4.2: mlflow.pytorch.log_model によるモデル保存とロード

前のセルですでに `mlflow.pytorch.log_model(model, "model")` でモデルを保存しました。  
ここでは、保存したモデルを `mlflow.pytorch.load_model()` でロードして推論を行います。

| API | 動作 |
|---|---|
| `mlflow.pytorch.log_model(model, artifact_path)` | モデルを `runs:/<run_id>/<artifact_path>` に保存 |
| `mlflow.pytorch.load_model(model_uri)` | `runs:/` や `models:/` URI からモデルをロード |

**sklearn との対応:**  
sklearn の `mlflow.sklearn.log_model()` / `mlflow.sklearn.load_model()` と同じパターンです。  
フレームワークが変わっても MLflow の API 設計は統一されています。

In [5]:
# runs:/ URI でモデルをロードして推論
model_uri = f"runs:/{run.info.run_id}/model"
loaded_model = mlflow.pytorch.load_model(model_uri)
loaded_model.eval()

print(f"モデルをロードしました: {model_uri}")
print(f"ロードしたモデルの型: {type(loaded_model)}")

with torch.no_grad():
    sample = X_test[:3]
    preds = loaded_model(sample).argmax(dim=1)
    print(f"Predictions: {preds.numpy()}")
    print(f"True labels: {y_test[:3].numpy()}")

モデルをロードしました: runs:/b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7/model
ロードしたモデルの型: <class '__main__.SimpleNet'>
Predictions: [1 2 0]
True labels: [2 1 0]


In [6]:
# MLflow クライアントで記録内容を確認
from mlflow import MlflowClient

client = MlflowClient()

exp = client.get_experiment_by_name("pytorch-classification")
runs = client.search_runs(experiment_ids=[exp.experiment_id])

print("=== 記録済み Run の確認 ===")
for r in runs:
    print(f"Run: {r.info.run_name} | ID: {r.info.run_id} | Status: {r.info.status}")

# 最初の Run の詳細を表示
latest_run = runs[0]
print("\n=== パラメータ ===")
for k, v in sorted(latest_run.data.params.items()):
    print(f"  {k}: {v}")

print("\n=== 最終メトリクス ===")
for k, v in sorted(latest_run.data.metrics.items()):
    print(f"  {k}: {v:.4f}")

=== 記録済み Run の確認 ===
Run: pytorch_simple_net | ID: b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7 | Status: FINISHED

=== パラメータ ===
  hidden_dim: 32
  lr: 0.01
  n_classes: 3
  n_epochs: 20
  n_features: 10

=== 最終メトリクス ===
  test_accuracy: 0.41
  test_loss: 1.0681
  train_loss: 0.9507


---

## 補足: PyTorch Lightning 環境での mlflow.pytorch.autolog()

> **注意:** 以下は PyTorch Lightning がインストールされている環境向けの補足説明です。  
> このチュートリアルでは PyTorch Lightning は使用しません。

PyTorch Lightning を使用している場合は、`mlflow.pytorch.autolog()` を呼び出すだけでトレーニングループのメトリクスを自動記録できます。

```python
import mlflow
import pytorch_lightning as pl

# autolog を有効化（PyTorch Lightning 専用）
mlflow.pytorch.autolog()

trainer = pl.Trainer(max_epochs=20)
trainer.fit(model, train_dataloader)
# → train_loss / val_loss などが自動的に MLflow に記録される
```

あるいは、Lightning の `Trainer` に `MLflowLogger` を渡す方法もあります:

```python
from pytorch_lightning.loggers import MLFlowLogger

mlf_logger = MLFlowLogger(
    experiment_name="pytorch-classification",
    tracking_uri="./mlruns",
)
trainer = pl.Trainer(max_epochs=20, logger=mlf_logger)
trainer.fit(model, train_dataloader)
```

**参考:** 純粋な PyTorch（本ノートブックの方法）では `mlflow.pytorch.autolog()` はトレーニングループを自動検出しないため、  
エポックごとに明示的に `mlflow.log_metric()` を呼び出す必要があります。

---

## まとめ

このノートブックでは以下を学びました:

| 内容 | 使用 API |
|---|---|
| 合成分類データの生成 | `torch.randn()`, `torch.randint()` |
| 2 層 MLP の定義 | `nn.Module`, `nn.Linear`, `nn.ReLU` |
| パラメータの記録 | `mlflow.log_params()` |
| エポックごとのメトリクス記録 | `mlflow.log_metric(key, value, step=epoch)` |
| モデルの保存 | `mlflow.pytorch.log_model()` |
| モデルのロードと推論 | `mlflow.pytorch.load_model()` |
| 記録内容の確認 | `MlflowClient.search_runs()` |

### sklearn サンプルとの共通点

- `mlflow.set_tracking_uri()` / `mlflow.set_experiment()` / `mlflow.start_run()` の使い方は同じ
- `log_model()` / `load_model()` のパターンはフレームワークが変わっても統一されている
- `step` 引数を使うことで時系列メトリクスを記録できるのが PyTorch 訓練の特徴

### 次のステップ

次のノートブックでは Optuna と MLflow を連携させたハイパーパラメータ最適化（HPO）と、
複数の試行結果を比較してベストモデルを Model Registry に登録する手順を学びます。

**次のノートブック:** [04_experiment_management/](../04_experiment_management/)